In [1]:
import os
import time
import json
from api_client import TradingDeskAPI
from database import save_snapshots

import requests
import numpy as np
import pandas as pd
from scipy.stats import norm
import matplotlib.pyplot as plt
from datetime import datetime, timezone
from scipy.interpolate import PchipInterpolator

from concurrent.futures import ThreadPoolExecutor

In [2]:
api = TradingDeskAPI()

markets = api.get_markets()

markets_df = pd.DataFrame(markets)

markets_df.head()

,id,question,conditionId,slug,resolutionSource,endDate,liquidity,startDate,image,icon,...,positionIds,oneDayPriceChange,oneHourPriceChange,sportsMarketType,gameId,eventStartTime,marketMetadata,oneYearPriceChange,umaResolutionStatus,line
0,1361063,Will Carolina Panthers win the 2027 NFL NFC Ch...,0x7be2cc6b32ca3b9487164a5e85cc890ef6f358ba010d...,will-carolina-panthers-win-the-2027-nfl-nfc-ch...,,2027-01-25T00:00:00Z,99971.89206,2026-02-09T22:31:13.404423Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1361088,Will Houston Texans win the 2027 NFL AFC Champ...,0x86f097adec2b38cbd44f02d1476a20b26a4ba82d00ce...,will-houston-texans-win-the-2027-nfl-afc-champ...,,2027-01-25T00:00:00Z,99842.64342,2026-02-09T22:34:08.624851Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3044036,Will AS Omónoia Leukosías win on 2026-08-05?,0xe99c72b02bb55fa9a2767980aba799b59409937dad14...,uwcl-ef-omo-2026-08-05-omo,https://www.uefa.com/womenschampionsleague/,2026-08-05T17:00:00Z,99519.45188,2026-07-22T20:00:30Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,[105130369626428534305239084062363133016845655...,-0.155,0.006,moneyline,NaN,NaN,NaN,NaN,NaN,NaN
3,1361091,Will Tennessee Titans win the 2027 NFL AFC Cha...,0xc536ab586b250f5928bf4da82030a642b1d193b95b38...,will-tennessee-titans-win-the-2027-nfl-afc-cha...,,2027-01-25T00:00:00Z,99288.32106,2026-02-09T22:34:29.047459Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,559699,Will Elissa Slotkin win the 2028 Democratic pr...,0xe50c71cc5e2372bf0f6194a02e51459dd03b2c7b2d78...,will-elissa-slotkin-win-the-2028-democratic-pr...,,2028-11-07T00:00:00Z,992811.65552,2025-07-11T18:36:44.703Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
BTC_KEYWORDS = [
    "bitcoin",
    "btc",
    "xbt",
    "bitcoin price",
    "bitcoin hits",
    "bitcoin above",
    "bitcoin below",
]

def is_btc_market(row):
    text = (str(row["question"]) + " " + str(row.get("description", ""))).lower()

    return any(k in text for k in BTC_KEYWORDS)

btc_df = markets_df[markets_df.apply(is_btc_market, axis=1)]
btc_df

,id,question,conditionId,slug,resolutionSource,endDate,liquidity,startDate,image,icon,...,positionIds,oneDayPriceChange,oneHourPriceChange,sportsMarketType,gameId,eventStartTime,marketMetadata,oneYearPriceChange,umaResolutionStatus,line
7,701486,"Will Bitcoin reach $200,000 by December 31, 2026?",0xac32e73aa9e0dae801d88d4f81efd2ef3fa0f04b815f...,will-bitcoin-reach-200000-by-december-31-2026-...,,2027-01-01T05:00:00Z,98980.65277,2025-11-24T19:07:20.933Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,NaN,0.0010,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10,701496,"Will Bitcoin reach $100,000 by December 31, 2026?",0xdaa4866bae18be58c5a79d2aeeffd035ec78f1bb49db...,will-bitcoin-reach-100000-by-december-31-2026-...,,2027-01-01T05:00:00Z,98287.0336,2025-11-24T19:07:17.691Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
48,2467210,"Will Bitcoin reach $70,000 by December 31, 2026?",0x7b9072e6a9cdcf022c4f098564e8ee612d544e179e4a...,will-bitcoin-reach-70000-by-december-31-2026-f...,,2027-01-01T05:00:00Z,92316.6797,2026-06-08T04:59:37.841638Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,NaN,-0.0050,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
64,3257370,"Will Bitcoin dip to $40,000 in August?",0x6d241c554cf546599063b5b87c12c5c9c569f7f9e482...,will-bitcoin-dip-to-40k-in-august-2026,NaN,2026-09-01T04:00:00Z,90552.36363,2026-08-01T05:07:38Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,NaN,-0.0005,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
88,3257362,"Will Bitcoin dip to $50,000 in August?",0xdf1463edbf062d8fdd82c5eb0bdb5fd69450b0c21c9d...,will-bitcoin-dip-to-50k-in-august-2026,NaN,2026-09-01T04:00:00Z,88561.14941,2026-08-01T05:07:37Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,NaN,-0.0050,-0.0015,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
for i in btc_df.columns:
    print(i)

id
question
conditionId
slug
resolutionSource
endDate
liquidity
startDate
image
icon
description
outcomes
outcomePrices
volume
active
closed
marketMakerAddress
createdAt
updatedAt
new
featured
submitted_by
archived
resolvedBy
restricted
groupItemTitle
groupItemThreshold
questionID
enableOrderBook
orderPriceMinTickSize
orderMinSize
volumeNum
liquidityNum
endDateIso
startDateIso
hasReviewedDates
clobTokenIds
comboStatus
umaBond
umaReward
volumeClob
liquidityClob
makerBaseFee
takerBaseFee
customLiveness
acceptingOrders
negRisk
negRiskMarketID
negRiskRequestID
events
ready
funded
acceptingOrdersTimestamp
cyom
competitive
pagerDutyNotificationEnabled
approved
rewardsMinSize
rewardsMaxSpread
spread
oneMonthPriceChange
lastTradePrice
bestBid
bestAsk
automaticallyActive
clearBookOnStart
seriesColor
showGmpSeries
showGmpOutcome
manualActivation
negRiskOther
umaResolutionStatuses
pendingDeployment
deploying
deployingTimestamp
rfqEnabled
holdingRewardsEnabled
feesEnabled
requiresTranslation
feeTy

In [ ]:
btc_df.iloc[0].clobTokenIds

In [5]:
btc_df = btc_df[
            (btc_df["acceptingOrders"] == True) &
            (btc_df["enableOrderBook"] == True)
        ]

btc_df["tokens"] = btc_df["clobTokenIds"].apply(json.loads)
btc_df["yes_token"] = btc_df["tokens"].apply(lambda x: x[0])
btc_df["no_token"] = btc_df["tokens"].apply(lambda x: x[1])

btc_df

,id,question,conditionId,slug,resolutionSource,endDate,liquidity,startDate,image,icon,...,sportsMarketType,gameId,eventStartTime,marketMetadata,oneYearPriceChange,umaResolutionStatus,line,tokens,yes_token,no_token
7,701486,"Will Bitcoin reach $200,000 by December 31, 2026?",0xac32e73aa9e0dae801d88d4f81efd2ef3fa0f04b815f...,will-bitcoin-reach-200000-by-december-31-2026-...,,2027-01-01T05:00:00Z,98980.65277,2025-11-24T19:07:20.933Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,[613689431282552874145652703368566154530006753...,6136894312825528741456527033685661545300067537...,9699347185440015640867052761315094444335927219...
10,701496,"Will Bitcoin reach $100,000 by December 31, 2026?",0xdaa4866bae18be58c5a79d2aeeffd035ec78f1bb49db...,will-bitcoin-reach-100000-by-december-31-2026-...,,2027-01-01T05:00:00Z,98287.0336,2025-11-24T19:07:17.691Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,[560789380600969764480867542494973004473603337...,5607893806009697644808675424949730044736033378...,1129166290489771317466790338838869664064361055...
48,2467210,"Will Bitcoin reach $70,000 by December 31, 2026?",0x7b9072e6a9cdcf022c4f098564e8ee612d544e179e4a...,will-bitcoin-reach-70000-by-december-31-2026-f...,,2027-01-01T05:00:00Z,92316.6797,2026-06-08T04:59:37.841638Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,[955325244079599031455105836849790497350285024...,9553252440795990314551058368497904973502850245...,1078699501271841417437724297387015787237705826...
64,3257370,"Will Bitcoin dip to $40,000 in August?",0x6d241c554cf546599063b5b87c12c5c9c569f7f9e482...,will-bitcoin-dip-to-40k-in-august-2026,NaN,2026-09-01T04:00:00Z,90552.36363,2026-08-01T05:07:38Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,[802297131074916049881662275754208115524036207...,8022971310749160498816622757542081155240362077...,4317110822368379292622911619394151317061935215...
88,3257362,"Will Bitcoin dip to $50,000 in August?",0xdf1463edbf062d8fdd82c5eb0bdb5fd69450b0c21c9d...,will-bitcoin-dip-to-50k-in-august-2026,NaN,2026-09-01T04:00:00Z,88561.14941,2026-08-01T05:07:37Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,[747390749095846764977874016868928588526100638...,7473907490958467649778740168689285885261006384...,9277048309813507514592692662054100880729326746...


In [6]:
def get_books(row):

    yes_book = api.get_orderbook(row.yes_token)
    no_book = api.get_orderbook(row.no_token)

    return pd.Series({
        "yes_book": yes_book,
        "no_book": no_book,

        "yes_bid": float(yes_book["bids"][-1]["price"]) if yes_book["bids"] else None,
        "yes_ask": float(yes_book["asks"][-1]["price"]) if yes_book["asks"] else None,

        "no_bid": float(no_book["bids"][-1]["price"]) if no_book["bids"] else None,
        "no_ask": float(no_book["asks"][-1]["price"]) if no_book["asks"] else None,
    })

# btc_df[
#     [
#         "yes_book",
#         "no_book",
#         "yes_bid",
#         "yes_ask",
#         "no_bid",
#         "no_ask"
#     ]
# ] = btc_df.apply(
#     get_books,
#     axis=1
# )

In [7]:
with ThreadPoolExecutor(max_workers=30) as executor:
    results = list(executor.map(get_books, [row for _, row in btc_df.iterrows()]))

btc_df[
    [
        "yes_book",
        "no_book",
        "yes_bid",
        "yes_ask",
        "no_bid",
        "no_ask"
    ]
] = pd.DataFrame(results).values

In [8]:
btc_df

,id,question,conditionId,slug,resolutionSource,endDate,liquidity,startDate,image,icon,...,line,tokens,yes_token,no_token,yes_book,no_book,yes_bid,yes_ask,no_bid,no_ask
7,701486,"Will Bitcoin reach $200,000 by December 31, 2026?",0xac32e73aa9e0dae801d88d4f81efd2ef3fa0f04b815f...,will-bitcoin-reach-200000-by-december-31-2026-...,,2027-01-01T05:00:00Z,98980.65277,2025-11-24T19:07:20.933Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,NaN,[613689431282552874145652703368566154530006753...,6136894312825528741456527033685661545300067537...,9699347185440015640867052761315094444335927219...,{'market': '0xac32e73aa9e0dae801d88d4f81efd2ef...,{'market': '0xac32e73aa9e0dae801d88d4f81efd2ef...,0.021,0.022,0.978,0.979
10,701496,"Will Bitcoin reach $100,000 by December 31, 2026?",0xdaa4866bae18be58c5a79d2aeeffd035ec78f1bb49db...,will-bitcoin-reach-100000-by-december-31-2026-...,,2027-01-01T05:00:00Z,98287.0336,2025-11-24T19:07:17.691Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,NaN,[560789380600969764480867542494973004473603337...,5607893806009697644808675424949730044736033378...,1129166290489771317466790338838869664064361055...,{'market': '0xdaa4866bae18be58c5a79d2aeeffd035...,{'market': '0xdaa4866bae18be58c5a79d2aeeffd035...,0.07,0.09,0.91,0.93
48,2467210,"Will Bitcoin reach $70,000 by December 31, 2026?",0x7b9072e6a9cdcf022c4f098564e8ee612d544e179e4a...,will-bitcoin-reach-70000-by-december-31-2026-f...,,2027-01-01T05:00:00Z,92316.6797,2026-06-08T04:59:37.841638Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,NaN,[955325244079599031455105836849790497350285024...,9553252440795990314551058368497904973502850245...,1078699501271841417437724297387015787237705826...,{'market': '0x7b9072e6a9cdcf022c4f098564e8ee61...,{'market': '0x7b9072e6a9cdcf022c4f098564e8ee61...,0.69,0.7,0.3,0.31
64,3257370,"Will Bitcoin dip to $40,000 in August?",0x6d241c554cf546599063b5b87c12c5c9c569f7f9e482...,will-bitcoin-dip-to-40k-in-august-2026,NaN,2026-09-01T04:00:00Z,90552.36363,2026-08-01T05:07:38Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,NaN,[802297131074916049881662275754208115524036207...,8022971310749160498816622757542081155240362077...,4317110822368379292622911619394151317061935215...,{'market': '0x6d241c554cf546599063b5b87c12c5c9...,{'market': '0x6d241c554cf546599063b5b87c12c5c9...,0.008,0.01,0.99,0.992
88,3257362,"Will Bitcoin dip to $50,000 in August?",0xdf1463edbf062d8fdd82c5eb0bdb5fd69450b0c21c9d...,will-bitcoin-dip-to-50k-in-august-2026,NaN,2026-09-01T04:00:00Z,88561.14941,2026-08-01T05:07:37Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,NaN,[747390749095846764977874016868928588526100638...,7473907490958467649778740168689285885261006384...,9277048309813507514592692662054100880729326746...,{'market': '0xdf1463edbf062d8fdd82c5eb0bdb5fd6...,{'market': '0xdf1463edbf062d8fdd82c5eb0bdb5fd6...,0.046,0.047,0.953,0.954


In [9]:
import re
from dateutil import parser


def extract_strike(question):

    match = re.search(
        r'\$([\d,]+)',
        question
    )

    if match:
        return float(
            match.group(1).replace(",", "")
        )

    return None

def extract_expiry(question):

    date = re.search(
        r'(January|February|March|April|May|June|July|August|September|October|November|December) \d{1,2}, \d{4}',
        question
    )

    if date:
        dt = parser.parse(date.group())

        return dt.strftime("%Y%m%d")

    return None

btc_df["strike"] = btc_df.question.apply(
    extract_strike
)

btc_df["expiry"] = btc_df.question.apply(
    extract_expiry
)

In [10]:
def parse_btc_market_type(question):

    q = question.lower()

    # ----------------------
    # Direction
    # ----------------------

    if any(word in q for word in [
        "dip",
        "fall",
        "drop",
        "below",
        "under",
        "crash"
    ]):
        direction = "down"


    elif any(word in q for word in [
        "reach",
        "hit",
        "touch",
        "above",
        "over",
        "exceed"
    ]):
        direction = "up"


    else:
        direction = None

    # ----------------------
    # Event type
    # ----------------------

    if any(word in q for word in [
        "reach",
        "hit",
        "touch",
        "dip to",
        "fall to",
        "drop to",
        "crash to"
    ]):
        event_type = "touch"


    elif any(word in q for word in [
        "be above",
        "above on",
        "close above",
        "be below",
        "below on",
        "close below",
        "finish above",
        "finish below"
    ]):
        event_type = "expiry"


    else:
        event_type = None


    return {
        "direction": direction,
        "event_type": event_type
    }

btc_df[
    ["direction", "event_type"]
] = btc_df["question"].apply(
    lambda x: pd.Series(parse_btc_market_type(x))
)

In [11]:
btc_df

,id,question,conditionId,slug,resolutionSource,endDate,liquidity,startDate,image,icon,...,yes_book,no_book,yes_bid,yes_ask,no_bid,no_ask,strike,expiry,direction,event_type
7,701486,"Will Bitcoin reach $200,000 by December 31, 2026?",0xac32e73aa9e0dae801d88d4f81efd2ef3fa0f04b815f...,will-bitcoin-reach-200000-by-december-31-2026-...,,2027-01-01T05:00:00Z,98980.65277,2025-11-24T19:07:20.933Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,{'market': '0xac32e73aa9e0dae801d88d4f81efd2ef...,{'market': '0xac32e73aa9e0dae801d88d4f81efd2ef...,0.021,0.022,0.978,0.979,200000.0,20261231,up,touch
10,701496,"Will Bitcoin reach $100,000 by December 31, 2026?",0xdaa4866bae18be58c5a79d2aeeffd035ec78f1bb49db...,will-bitcoin-reach-100000-by-december-31-2026-...,,2027-01-01T05:00:00Z,98287.0336,2025-11-24T19:07:17.691Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,{'market': '0xdaa4866bae18be58c5a79d2aeeffd035...,{'market': '0xdaa4866bae18be58c5a79d2aeeffd035...,0.07,0.09,0.91,0.93,100000.0,20261231,up,touch
48,2467210,"Will Bitcoin reach $70,000 by December 31, 2026?",0x7b9072e6a9cdcf022c4f098564e8ee612d544e179e4a...,will-bitcoin-reach-70000-by-december-31-2026-f...,,2027-01-01T05:00:00Z,92316.6797,2026-06-08T04:59:37.841638Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,{'market': '0x7b9072e6a9cdcf022c4f098564e8ee61...,{'market': '0x7b9072e6a9cdcf022c4f098564e8ee61...,0.69,0.7,0.3,0.31,70000.0,20261231,up,touch
64,3257370,"Will Bitcoin dip to $40,000 in August?",0x6d241c554cf546599063b5b87c12c5c9c569f7f9e482...,will-bitcoin-dip-to-40k-in-august-2026,NaN,2026-09-01T04:00:00Z,90552.36363,2026-08-01T05:07:38Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,{'market': '0x6d241c554cf546599063b5b87c12c5c9...,{'market': '0x6d241c554cf546599063b5b87c12c5c9...,0.008,0.01,0.99,0.992,40000.0,NaN,down,touch
88,3257362,"Will Bitcoin dip to $50,000 in August?",0xdf1463edbf062d8fdd82c5eb0bdb5fd69450b0c21c9d...,will-bitcoin-dip-to-50k-in-august-2026,NaN,2026-09-01T04:00:00Z,88561.14941,2026-08-01T05:07:37Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,{'market': '0xdf1463edbf062d8fdd82c5eb0bdb5fd6...,{'market': '0xdf1463edbf062d8fdd82c5eb0bdb5fd6...,0.046,0.047,0.953,0.954,50000.0,NaN,down,touch


In [ ]:
def save(df):

    date = datetime.now().strftime("%Y%m%d")
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    DATA_DIR = f"data/{date}"
    os.makedirs(DATA_DIR, exist_ok=True)
    filename = f"{DATA_DIR}/markets_{timestamp}.parquet"

    df.to_parquet(filename, engine="fastparquet", index=False)

    print('df saved at: ', filename)

In [ ]:
save(btc_df)

In [13]:
def calculate_ev(
        model_prob,
        best_ask_yes,
        best_bid_yes,
        best_ask_no,
        best_bid_no,
        fee_rate=0.07,
    ):
    """
    Expected PnL per contract for Polymarket.

    model_prob : P(YES)
    fee_rate   : taker fee rate (e.g. 0.07 for crypto markets)

    Assumes:
      - you are a taker
      - fee = fee_rate * price * (1 - price)
      - settlement has no fee
    """

    def is_valid(price):
        return pd.notna(price)

    p_yes = model_prob
    p_no = 1 - p_yes

    def fee(price):
        #fee = C × feeRate × p × (1 - p)
        #Where C = number of shares traded and p = price of the shares.
        return fee_rate * price * (1 - price)

    # -------------------------
    # BUY YES
    # -------------------------
    if is_valid(best_ask_yes):
        buy_yes_cost = best_ask_yes + fee(best_ask_yes)

        profit_if_yes = 1 - buy_yes_cost
        cost_if_no = buy_yes_cost

        # EV: profit if yes - cost if no
        buy_yes_ev = p_yes * profit_if_yes - p_no * cost_if_no

        # -------------------------
        # SELL YES (short YES)
        # -------------------------
        sell_yes_credit = best_bid_yes - fee(best_bid_yes)

        profit_if_no = sell_yes_credit
        cost_if_yes = 1 - sell_yes_credit # need to pay the remaining of the $1 out of the credit you got, if yes happened

        # EV: profit if yes - cost if no
        sell_yes_ev = p_no * profit_if_no - p_yes * cost_if_yes

    else:
        buy_yes_ev = np.nan
        sell_yes_ev = np.nan
        
    # -------------------------
    # BUY NO
    # -------------------------
    if is_valid(best_ask_no):
        buy_no_cost = best_ask_no + fee(best_ask_no)

        profit_if_no = 1 - buy_no_cost
        cost_if_yes = buy_no_cost

        buy_no_ev = p_no * profit_if_no - p_yes * cost_if_yes
    

        # -------------------------
        # SELL NO (short NO)
        # -------------------------
        sell_no_credit = best_bid_no - fee(best_bid_no)

        profit_if_yes = sell_no_credit
        cost_if_no = 1 - sell_no_credit

        sell_no_ev = p_yes * profit_if_yes - p_no * cost_if_no

    else:
        buy_no_ev = np.nan
        sell_no_ev = np.nan


    print("buy_yes_ev:", buy_yes_ev)
    print("sell_yes_ev:", sell_yes_ev)
    print("buy_no_ev:", buy_no_ev)
    print("sell_no_ev:", sell_no_ev)

    return {
        "buy_yes_ev": buy_yes_ev,
        "sell_yes_ev": sell_yes_ev,
        "buy_no_ev": buy_no_ev,
        "sell_no_ev": sell_no_ev,
        "buy_yes_fee": fee(best_ask_yes),
        "sell_yes_fee": fee(best_bid_yes),
        "buy_no_fee": fee(best_ask_no),
        "sell_no_fee": fee(best_bid_no),
    }


result = calculate_ev(
    model_prob=0.059,
    best_ask_yes=0.1,
    best_bid_yes=0.09,
    best_ask_no=0.91,
    best_bid_no=0.9,
    fee_rate=0.07
)

# for k,v in result.items():
#     print(k, v)

buy_yes_ev: -0.04730000000000002
sell_yes_ev: 0.025266999999999998
buy_no_ev: 0.025266999999999984
sell_no_ev: -0.04729999999999996


In [ ]:
def calculate_market_ev(row, btc_surface):

    strike = row["strike"]

    T = row["T"]

    iv = btc_surface.get_iv_from_surface(
        strike,
        T
    )

    if iv is None:
        return None

    if row["event_type"] == "touch":

        p_touch_above, p_touch_below = btc_surface.prob_touch(
                    btc_surface.weighted_spot,
                    strike,
                    iv,
                    T
                )
        
        if row["direction"] == "up":
            model_prob = p_touch_above

        elif row["direction"] == "down":
             model_prob = p_touch_below

    if row["event_type"] == "expiry":

        p_finish_above, p_finish_below = btc_surface.prob_finish(
            btc_surface.weighted_spot,
            strike,
            iv,
            T
        )

        if row["direction"] == "up":
            model_prob = p_finish_above

        elif row["direction"] == "down":
            model_prob = p_finish_below

    ev = calculate_ev(
        model_prob=model_prob,
        best_ask_yes=row["yes_ask"],
        best_bid_yes=row["yes_bid"],
        best_ask_no=row["no_ask"],
        best_bid_no=row["no_bid"],
    )

    return pd.Series({
        "iv": iv,
        "model_prob": model_prob,
        **ev
    })

results = btc_df.apply(
    lambda x: calculate_market_ev(
        x,
        variance_surface
    ),
    axis=1
)

btc_df = pd.concat(
    [
        btc_df,
        results
    ],
    axis=1
)

In [ ]:
ev_df = btc_df.apply(
    calculate_market_ev,
    axis=1
)

btc_df = pd.concat(
    [
        btc_df,
        ev_df
    ],
    axis=1
)

In [ ]:
ENTRY_THRESHOLD = 0.03
EXIT_THRESHOLD = 0.01

edge = 0.031

if edge > ENTRY_THRESHOLD:
    pass
    #sell_yes()

if edge < EXIT_THRESHOLD:
    pass
    #close_position()

In [ ]:
#                  Polymarket    Model (exchanges: deribit, bybit, okx)    Edge

# 80k Dec-26        22%          20%       +2%
# 90k Dec-26        14%          12%       +2%
# 100k Dec-26        9.5%         5.9%     +3.6%
# 110k Dec-26        7%           4%       +3%
# 120k Dec-26        5%           2%       +3%

In [ ]:
# Keep trading journal

# Your thesis.
# Why you think the market is mispriced.
# Position size.
# Exit criteria.
# What actually happened.

In [ ]:
# Stage 2 — Semi-automated execution

# The bot does:

# find opportunities
# calculate size
# prepare orders

# You approve:

# BUY YES
# Market: BTC above $150k
# Price: 0.43
# Size: $500
# Expected edge: +8%

# Click "confirm".

# This is useful because prediction markets can have:

# ambiguous wording
# resolution risks
# sudden news events

In [ ]:
# Day 1 — API connection + market ingestion

# Goal:

# Can I pull markets automatically?

# Build:

# get_markets()

# Output:

# {
#  "condition_id": "...",
#  "question": "Will X happen?",
#  "tokens": [
#     {
#       "token_id": "YES",
#       "price": 0.42
#     },
#     {
#       "token_id": "NO",
#       "price": 0.58
#     }
#  ]
# }

# Store:

# markets

# condition_id
# question
# yes_token
# no_token
# created_time
# Day 2 — Historical snapshots

# Goal:

# Can I reconstruct what the market looked like yesterday?

# Every 5 minutes:

# while True:

#     markets = api.get_markets()

#     for m in markets:
#         database.save_snapshot(m)

#     sleep(300)

# Database:

# market_snapshots

# timestamp
# condition_id
# yes_price
# no_price
# volume
# Day 3 — Order book collection

# Now collect microstructure data.

# For selected markets:

# get_orderbook(token_id)

# Store:

# orderbook_snapshots

# timestamp

# token_id

# best_bid
# best_ask

# bid_depth
# ask_depth

# Calculate:

# Spread
# spread=ask−bid

# Example:

# Bid:
# 0.42

# Ask:
# 0.46

# Spread:
# 4 cents
# Day 4 — Build your scanner

# Your first scanner should be dumb but useful.

# Signal 1: Large moves
# if abs(price_change_24h) > 0.10:
#     flag()

# Example:

# AI model release

# Yesterday:
# 35%

# Today:
# 52%

# Move:
# +17%
# Signal 2: Liquidity opportunities
# if spread > 0.08:
#     flag()
# Signal 3: Volume spikes
# if volume_today > 5 * average_volume:
#     flag()

# Your output:

# TOP MARKETS TO REVIEW

# 1.
# Question:
# Will Fed cut rates?

# Price:
# 42%

# 24h move:
# +12%

# Reason:
# Large movement


# 2.
# Question:
# Will Company X acquire Y?

# Price:
# 33%

# Spread:
# 11 cents

# Reason:
# Wide market
# Day 5 — Add your probability workflow

# Do NOT automate this yet.

# Create a manual table:

# trade_journal.csv

# market,current_price,my_probability,edge,reason
# Fed cut,0.42,0.55,0.13,"Inflation falling"
# AI launch,0.35,0.45,0.10,"Company comments"

# The key question:

# Market probability:
# 42%

# My probability:
# 55%

# Difference:
# +13%
# Day 6 — Paper trading

# Before your C++ engine touches anything:

# Create:

# paper_buy(
#     market,
#     price,
#     size
# )

# Track:

# Position:
# YES Fed cut

# Entry:
# 42c

# Size:
# $100

# Current:
# 48c

# P&L:
# +$14
# Day 7 — Analytics

# Calculate:

# Return
# profit / capital
# Drawdown

# Largest loss from peak.

# Sortino

# Track:

# returns
# negative returns only

# Your output:

# Paper Portfolio

# Trades:
# 18

# Win rate:
# 61%

# Return:
# +8.4%

# Max drawdown:
# -2.1%

# Sortino:
# 2.4

In [ ]:
{'id': '701496', 'question': 'Will Bitcoin reach $100,000 by December 31, 2026?', 'conditionId': '0xdaa4866bae18be58c5a79d2aeeffd035ec78f1bb49dbd88f72993997778a990f', 'slug': 'will-bitcoin-reach-100000-by-december-31-2026-571-361-361', 
 'resolutionSource': '', 'endDate': '2027-01-01T05:00:00Z', 'liquidity': '94083.1351', 'startDate': '2025-11-24T19:07:17.691Z', 'image': 'https://polymarket-upload.s3.us-east-2.amazonaws.com/BTC+fullsize.png', 
 'icon': 'https://polymarket-upload.s3.us-east-2.amazonaws.com/BTC+fullsize.png', 
'description': 'This market will immediately resolve to "Yes" if any Binance 1 minute candle for Bitcoin (BTC/USDT) between November 24, 2025, 14:00 and December 31, 2026, 23:59 in the ET timezone has a final "High" price equal to or greater than the price specified in the title. Otherwise, this market will resolve to "No."\n\nThe resolution source for this market is Binance, specifically the BTC/USDT "High" prices available at https://www.binance.com/en/trade/BTC_USDT, with the chart settings on "1m" for one-minute candles selected on the top bar.\n\nPlease note that the outcome of this market depends solely on the price data from the Binance BTC/USDT trading pair. Prices from other exchanges, different trading pairs, or spot markets will not be considered for the resolution of this market.', 
'outcomes': '["Yes", "No"]', 'outcomePrices': '["0.095", "0.905"]', 'volume': '2370982.5583320004', 'active': True, 'closed': False, 'marketMakerAddress': '', 'createdAt': '2025-11-24T18:55:12.725029Z', 'updatedAt': '2026-08-02T10:54:52.526927Z', 
'new': False, 'featured': False, 'submitted_by': '0x91430CaD2d3975766499717fA0D66A78D814E5c5', 'archived': False, 'resolvedBy': '0x65070BE91477460D8A7AeEb94ef92fe056C2f2A7', 'restricted': True, 'groupItemTitle': '↑ 100,000', 'groupItemThreshold': '13', 
'questionID': '0x3c9be67d4b90291760ac3bffc1f9470a1966e5c1f3e99131333170e3469bd023', 'enableOrderBook': True, 'orderPriceMinTickSize': 0.01, 'orderMinSize': 5, 'volumeNum': 2370982.5583320004, 'liquidityNum': 94083.1351, 'endDateIso': '2027-01-01', 
'startDateIso': '2025-11-24', 'hasReviewedDates': True, 'volume24hr': 1365.610655, 'volume1wk': 66640.791618, 'volume1mo': 228733.97684700004, 'volume1yr': 2370982.5583320004, 
'clobTokenIds': '["56078938060096976448086754249497300447360333783952000147427828224794011030104", "11291662904897713174667903388388696640643610556195928998276904135282270136756"]', 
'comboStatus': 'disabled', 'umaBond': '500', 'umaReward': '5', 'volume24hrClob': 1365.610655, 'volume1wkClob': 66640.791618, 'volume1moClob': 228733.97684700004, 'volume1yrClob': 2370982.5583320004, 'volumeClob': 2370982.5583320004, 
'liquidityClob': 94083.1351, 'makerBaseFee': 1000, 'takerBaseFee': 1000, 'customLiveness': 0, 'acceptingOrders': True, 'negRisk': False, 'negRiskRequestID': '', 
'events': [{'id': '89502', 'ticker': 'what-price-will-bitcoin-hit-before-2027', 'slug': 'what-price-will-bitcoin-hit-before-2027', 
            'title': 'What price will Bitcoin hit in 2026?', 'description': 'What price will Bitcoin hit before 2027?  ', 
            'resolutionSource': '', 'startDate': '2025-11-24T19:07:12.848Z', 'creationDate': '2025-11-24T19:13:13.705687Z', 'endDate': '2027-01-01T05:00:00Z', 
            'image': 'https://polymarket-upload.s3.us-east-2.amazonaws.com/BTC+fullsize.png', 'icon': 'https://polymarket-upload.s3.us-east-2.amazonaws.com/BTC+fullsize.png', 
            'active': True, 'closed': False, 'archived': False, 'new': False, 'featured': False, 'restricted': True, 'liquidity': 2695251.24076, 'volume': 50935482.262915, 
            'openInterest': 9503925.568983998, 'createdAt': '2025-11-24T18:55:05.597959Z', 'updatedAt': '2026-08-02T10:55:09.654318Z', 'competitive': 0.9999750006249843, 
            'volume24hr': 130499.32620200001, 'volume1wk': 2062901.9714630004, 'volume1mo': 7099616.726362999, 'volume1yr': 49102712.92022599, 'enableOrderBook': True, 
            'liquidityClob': 2695251.24076, 'negRisk': False, 'commentCount': 0, 'series': [{'id': '10016', 'ticker': 'bitcoin-hit-price-monthly', 'slug': 'bitcoin-hit-price-monthly', 
                                                                                        'title': 'Bitcoin Hit Price Monthly', 'seriesType': 'single', 'recurrence': 'monthly', 
                                                                                        'image': 'https://polymarket-upload.s3.us-east-2.amazonaws.com/bitcoin+colors.jpeg', 
                                                                                        'icon': 'https://polymarket-upload.s3.us-east-2.amazonaws.com/bitcoin+colors.jpeg', 
                                                                                        'active': True, 'closed': False, 'archived': False, 'featured': False, 'restricted': True, 
                                                                                        'createdAt': '2025-01-31T22:03:50.00441Z', 'updatedAt': '2026-08-02T10:55:28.185077Z', 
                                                                                        'volume24hr': 601000.637578, 'volume': 51492794.133888, 'liquidity': 3438609.2653, 'commentCount': 6318, 
                                                                                        'requiresTranslation': False}], 
            'cyom': False, 'showAllOutcomes': True, 'showMarketImages': False, 'enableNegRisk': False, 'automaticallyActive': True, 'seriesSlug': 'bitcoin-hit-price-monthly', 
            'gmpChartMode': 'default', 'negRiskAugmented': False, 'estimateValue': True, 'cantEstimate': True, 'cumulativeMarkets': False, 'pendingDeployment': False, 'deploying': False, 
            'requiresTranslation': False, 'eventMetadata': {'context_requires_regen': True}, 'version': 'v1'}], 

'ready': False, 'funded': False, 'acceptingOrdersTimestamp': '2025-11-24T19:06:55Z', 
'cyom': False, 'competitive': 0.8590880780051975, 'pagerDutyNotificationEnabled': False, 'approved': True, 'clobRewards': [{'id': '418394', 'conditionId': '0xdaa4866bae18be58c5a79d2aeeffd035ec78f1bb49dbd88f72993997778a990f', 
                                                                                                                        'assetAddress': '0xc011a7e12a19f7b1f670d46f03b03f3342e82dfb', 'rewardsAmount': 0, 'rewardsDailyRate': 0.001, 
                                                                                                                        'startDate': '2026-06-03', 'endDate': '2500-12-31'}], 
'rewardsMinSize': 0, 'rewardsMaxSpread': 0, 'spread': 0.01, 'oneMonthPriceChange': -0.01, 'lastTradePrice': 0.09, 'bestBid': 0.09, 'bestAsk': 0.1, 'automaticallyActive': True, 
'clearBookOnStart': True, 'seriesColor': '', 'showGmpSeries': False, 'showGmpOutcome': False, 'manualActivation': False, 'negRiskOther': False, 'umaResolutionStatuses': '[]', 
'pendingDeployment': False, 'deploying': False, 'deployingTimestamp': '2025-11-24T19:06:23.727362Z', 'rfqEnabled': False, 'holdingRewardsEnabled': True, 'feesEnabled': True, 
'requiresTranslation': False, 'feeType': 'crypto_fees_v2', 'feeSchedule': {'exponent': 1, 'rate': 0.07, 'takerOnly': True, 'rebateRate': 0.2}, 'version': 'v1'}
["0.095", "0.905"]